In [ ]:
import datetime
import time
import json
import pandas as pd
import requests
from urllib.parse import urlparse
from github import Github, GithubException, UnknownObjectException
from pathlib import Path
import concurrent.futures
import threading
import time
from github import Github, Auth
from itertools import cycle 
from random import uniform

In [ ]:
df_cve = pd.read_csv("NVD_Data/cve_data.csv")  # Replace with your file path

df_cve = df_cve.dropna(subset=['severity'])

# Select first 4 and last 4 columns, but remove the 3rd column
selected_cols = list(df_cve.columns[:2]) + list(df_cve.columns[3:6]) + list(df_cve.columns[-2:])
subset_df = df_cve[selected_cols]

# Display as a table
from IPython.display import display
display(subset_df)

In [ ]:
def clean_json(value):
    """
    Cleans and parses JSON strings from DataFrame columns.
    
    Args:
        value: The value to be cleaned (typically a string containing JSON)
    
    Returns:
        - Parsed JSON as a list if successful
        - None for non-string values or invalid JSON
    """
    if not isinstance(value, str) or value.strip() in ('', 'nan', 'None'):
        return None
    
    try:
        # Handle common JSON formatting issues
        value = value.strip()
        value = value.replace("'", '"')  # Replace single quotes
        value = value.replace('None', 'null')  # Replace Python None with JSON null
        value = value.replace('True', 'true').replace('False', 'false')  # Fix boolean values
        
        parsed = json.loads(value)
        return parsed if isinstance(parsed, list) else None
    except (json.JSONDecodeError, TypeError, AttributeError):
        return None

In [ ]:
def extract_github_urls(reference_list):
    """
    Returns list of dictionaries with original and base URLs
    """
    results = []
    if isinstance(reference_list, list):
        for item in reference_list:
            if isinstance(item, dict) and 'url' in item:
                url = item['url'].strip()
                if 'github.com' in url.lower() and urlparse(url).scheme in ['http', 'https']:
                    parsed = urlparse(url)
                    path_parts = parsed.path.strip('/').split('/')
                    if len(path_parts) >= 2:
                        results.append({
                            'original_url': url,
                            'base_url': f"{parsed.scheme}://{parsed.netloc}/{path_parts[0]}/{path_parts[1]}"
                        })
    return results

In [ ]:
#df_cve['cleaned_refs'] = df_cve['reference_json'].apply(clean_json)
#all_repo_urls = df_cve['cleaned_refs'].apply(extract_github_urls).explode().dropna().unique()

#print(f"Found {len(all_repo_urls)} unique GitHub repositories")

#pd.DataFrame(all_repo_urls, columns=['repo_url']).to_csv('github_repositories.csv', index=False)

In [ ]:
from github import Github, Auth
from itertools import cycle

github_tokens = [t.strip() for t in os.getenv("GITHUB_TOKENS", "").split(",") if t.strip()]
if not github_tokens:
    raise RuntimeError("No GitHub tokens provided. Set GITHUB_TOKENS as comma-separated.")





In [ ]:
from github import Github, Auth

valid_tokens = []
for t in github_tokens:
    try:
        g = Github(auth=Auth.Token(t))
        _ = g.get_rate_limit()  # safe test call; no username lookup
        valid_tokens.append(t)
    except Exception:
        pass

if not valid_tokens:
    raise RuntimeError("No valid GitHub tokens found in GITHUB_TOKENS.")

# Overwrite the original list with valid tokens only
github_tokens = valid_tokens

# Stop execution if no tokens are valid
if not github_tokens:
    raise ValueError("No valid GitHub tokens found! Exiting...")


In [ ]:
import itertools
# Function to extract owner and repo from GitHub URL
def extract_github_owner_repo(repo_url):
    parsed_url = urlparse(repo_url)
    path_parts = parsed_url.path.strip('/').split('/')
    
    if len(path_parts) < 2:
        return None, None  # Invalid URL
    owner, repo = path_parts[0], path_parts[1]
    return owner, repo.rstrip('.git')  # Remove .git suffix if present

def find_available_github_repos(df_cve, csv_path, github_tokens, max_workers=5, verbose=False):

    token_cycle = cycle(github_tokens)
    token_lock = threading.Lock()
    
    
    # Clean and extract URLs
    df_cve['cleaned_refs'] = df_cve['reference_json'].apply(clean_json)
    
    # Get list of unique URLs with their base repos
    seen_urls = set()
    url_info_list = []
    
    for url_list in df_cve['cleaned_refs'].apply(extract_github_urls):
        for url_info in url_list:
            if url_info['original_url'] not in seen_urls:
                seen_urls.add(url_info['original_url'])
                url_info_list.append(url_info)
    
    if verbose:
        print(f"Found {len(url_info_list)} unique GitHub URLs to check")


# Step 3: Check repos in threads
    def check_repo(url_info):
        owner, repo = extract_github_owner_repo(url_info['base_url'])
        if not owner or not repo:
            return None

        retries = 3
        for attempt in range(retries):
            with token_lock:
                token = next(token_cycle)
            g = Github(auth=Auth.Token(token))
            try:
                g.get_repo(f"{owner}/{repo}")
                return url_info['original_url']
            except Exception as e:
                msg = str(e).lower()
                if "rate limit" in msg:
                    if verbose:
                        print(f"⏳ Rate limited. Retrying ({attempt+1}/{retries})...")
                    time.sleep(30)
                    continue
                elif "bad credentials" in msg:
                    if verbose:
                        print(f"⚠️ Token failed mid-run: {token[:6]}")
                    continue
                if verbose:
                    print(f"Repository unavailable: {url_info['base_url']} - {str(e)[:100]}")
                return None
        return None

    available_repos = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        try:
            results = executor.map(check_repo, url_info_list)
            available_repos = [url for url in results if url]
        except KeyboardInterrupt:
            if verbose:
                print("\nInterrupted - saving partial results")

    # Step 4: Save results
    csv_path = Path(csv_path)
    csv_path.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(available_repos, columns=['Available URLs']).to_csv(csv_path / 'available_repos.csv', index=False)

    if verbose:
        print(f"\n✅ Completed! Found {len(available_repos)} accessible repositories")
    return available_repos

In [ ]:
#available_repos = find_available_github_repos(df_cve, "data/", github_tokens, max_workers=5, verbose=True)

In [ ]:
available_repos_df = pd.DataFrame(available_repos)

# Define the directory path
csv_path = Path("Data")
csv_file_path = csv_path / "available_repos.csv"

# Save the DataFrame to a CSV file
available_repos_df.to_csv(csv_file_path, index=False)

In [ ]:
from github.GithubException import BadCredentialsException, UnknownObjectException, GithubException
import itertools

# Function to extract owner and repo from GitHub URL
def extract_github_owner_repo(repo_url):
    parsed_url = urlparse(repo_url)
    path_parts = parsed_url.path.strip('/').split('/')
    
    if len(path_parts) < 2:
        return None, None  # Invalid URL
    owner, repo = path_parts[0], path_parts[1]
    return owner, repo.rstrip('.git')  # Remove .git suffix if present

def fetch_commit_patches(git_link, repo, commit_sha):
    """Fetch the patch/diff for a specific commit"""
    try:
        commit = repo.get_commit(commit_sha)
        return commit.patch
    except Exception as e:
        print(f"Error fetching patch for commit {commit_sha}: {e}")
        return None

def fetch_code_from_commit(repo, commit_sha, file_path):
    """Fetch file content at specific commit"""
    try:
        contents = repo.get_contents(file_path, ref=commit_sha)
        return contents.decoded_content.decode('utf-8')
    except Exception as e:
        print(f"Error fetching {file_path} at {commit_sha}: {e}")
        return None
    
    
def find_security_fixes(git_link, repo, start_date=None):
    """Identify potential security fixes using commit message heuristics"""
    query = f'repo:{repo.full_name} is:pr is:merged label:security'
    if start_date:
        query += f' merged:>={start_date.isoformat()}'
    
    issues = git_link.search_issues(query)
    security_fixes = []
    
    for issue in issues:
        if issue.pull_request:
            pr = repo.get_pull(issue.number)
            if pr.merged:
                security_fixes.append({
                    'issue': issue,
                    'pr': pr,
                    'commit': pr.merge_commit_sha,
                    'files_changed': pr.changed_files
                })
    return security_fixes

def process_repository(git_link, repo_url):
    
    """Process a single repository to find vulnerable/patched pairs"""
    
    print(f"\n[>] Starting processing: {repo_url}")
    owner, project = extract_github_owner_repo(repo_url)
    if not owner or not project:
        print(f"Invalid GitHub URL format: {repo_url}")
        print(f"[!] Skipped invalid GitHub URL: {repo_url}")
        return None
    
    try:
        print(f"[>] Fetching repo: {owner}/{project}")
        repo = git_link.get_repo(f"{owner}/{project}")
        print(f"[✓] Repo found: {repo.full_name}")

        
        # Get metadata
        metadata = {
            'repo_url': repo_url,
            'repo_name': repo.full_name,
            'description': repo.description,
            'date_created': repo.created_at,
            'date_last_push': repo.pushed_at,
            'homepage': repo.homepage,
            'repo_language': repo.language,
            'forks_count': repo.forks,
            'stars_count': repo.stargazers_count,
            'owner': owner
        }
        
        # Find security-related fixes
        print(f"[>] Looking for security fixes...")
        security_fixes = find_security_fixes(repo, start_date=repo.created_at)

        print(f"[✓] Found {len(security_fixes)} possible security PRs.")
        
        patches = []
        for fix in security_fixes:
            commit = repo.get_commit(fix['commit'])
            
            # Get parent commit (vulnerable version)
            parent_commit = commit.parents[0] if commit.parents else None
            
            # Process each changed file
            for file in commit.files:
                if not (file.filename.endswith('.py') or file.filename.endswith('.java') or 
                       file.filename.endswith('.c') or file.filename.endswith('.cpp')): 
                    continue  # Skip non-code files
                
                # Get vulnerable and patched versions
                vulnerable_code = fetch_code_from_commit(repo, parent_commit.sha, file.filename) if parent_commit else None
                patched_code = fetch_code_from_commit(repo, commit.sha, file.filename)
                
                if vulnerable_code and patched_code:
                    patches.append({
                        'repo_url': repo_url,
                        'commit_sha': commit.sha,
                        'parent_sha': parent_commit.sha,
                        'filename': file.filename,
                        'patch': file.patch,
                        'vulnerable_code': vulnerable_code,
                        'patched_code': patched_code,
                        'commit_message': commit.commit.message,
                        'commit_date': commit.commit.author.date,
                        'fix_issue': fix['issue'].title if fix else None
                    })
        
        return {
            'metadata': metadata,
            'patches': patches
        }
        
    except UnknownObjectException:
        print(f"Repository {repo_url} not found (404). It may be private or deleted.")
    except GithubException as e:
        print(f"GitHub API error for {repo_url}: {e}")
        if "rate limit" in str(e).lower():
            reset_time = git_link.get_rate_limit().core.reset
            sleep_time = (reset_time - datetime.now()).total_seconds() + 10
            print(f"Rate limit hit. Sleeping for {sleep_time} seconds.")
            time.sleep(sleep_time)
            return process_repository(git_link, repo_url)  # Retry
    except Exception as e:
        print(f"Unexpected error for {repo_url}: {e}")
    return None

def fetch_github_patches(available_repo_urls, token_list):
    token_cycle = itertools.cycle(token_list)

    def _threaded_process_repository(repo_url):
        token = next(token_cycle)
        print(f"[🔑] Using token: {token[:5]}... for {repo_url}")
        git_link = Github(auth=Auth.Token(token))
        return process_repository(git_link, repo_url)
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        futures = [executor.submit(_threaded_process_repository, url) for url in available_repo_urls]


        results = [f.result() for f in concurrent.futures.as_completed(futures)]
    
    # Process results into DataFrames
    metadata_list = []
    patches_list = []
    
    for result in results:
        if result:
            metadata_list.append(result['metadata'])
            patches_list.extend(result['patches'])
    
    # Create DataFrames
    repo_columns = [
        'repo_url', 'repo_name', 'description', 'date_created', 'date_last_push',
        'homepage', 'repo_language', 'forks_count', 'stars_count', 'owner'
    ]
    df_metadata = pd.DataFrame(metadata_list, columns=repo_columns)
    
    patch_columns = [
        'repo_url', 'commit_sha', 'parent_sha', 'filename', 'patch',
        'vulnerable_code', 'patched_code', 'commit_message', 'commit_date', 'fix_issue'
    ]
    df_patches = pd.DataFrame(patches_list, columns=patch_columns)
    
    print(f'Collected metadata for {len(df_metadata)} repositories and {len(df_patches)} patches.')
    return df_metadata, df_patches

In [ ]:
csv_path = Path("Data")
csv_file_path = csv_path / "available_urls.csv"
print(csv_file_path)

In [ ]:
# Load the CSV file
available_urls_df = pd.read_csv(csv_file_path)

available_urls = available_urls_df.iloc[:, 0].dropna().tolist()

print(f"Processing {len(available_urls)} repositories...")
#for url in available_urls:
    #print(" -", url)

df_metadata, df_patches = fetch_github_patches(available_urls, github_tokens)

